## Configuración del Entorno e Importaciones

In [ ]:
import os
import random
import time
import urllib.parse
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
import torchaudio 
import warnings

from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer

# Configurar determinismo para asegurar reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Configuración de rutas relativas (subiendo un nivel desde la carpeta notebooks/)
BASE_DIR = "../dataset_aves"
METADATA_PATH = os.path.join(BASE_DIR, "df_metadata_audios.csv")
LABELS_PATH = os.path.join(BASE_DIR, "df_etiquetas_fuertes.csv")

# Crear los directorios base si no existen físicamente en la máquina
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs("../artifacts", exist_ok=True)

print("Inicialización completa.")
print(f"Ruta absoluta del dataset de aves: {os.path.abspath(BASE_DIR)}")

# Silenciar logs internos de compilación de TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# Ignorar advertencias de descarte de módulos de TensorFlow
warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow")

Inicialización completa.
Ruta absoluta del dataset de aves: /home/els4nchez/Pictures/Bird-Recognition-using-BioSED-model/dataset_aves


# BioSED: Notebook 1 - Adquisición de Datos y Pseudo-Etiquetado Temporal

## 1. Introducción y Contexto Metodológico

Este proyecto aborda el problema de **Detección de Eventos Sonoros Bioacústicos** (BioSED - *Bioacoustic Sound Event Detection*). A diferencia de la clasificación de audio estándar, donde se predice una única etiqueta global por clip, BioSED formula un problema más complejo: **identificar simultáneamente qué especie vocaliza y en qué intervalos temporales exactos ocurre la vocalización** (clasificación frame-a-frame).

Desde la perspectiva del Aprendizaje Estadístico, este problema se modela como un sistema débilmente supervisado de múltiples etiquetas en el tiempo. La adquisición de anotaciones manuales detalladas (segmentos fuertes hechos por humanos) es altamente costosa. Por lo tanto, implementaremos un flujo de trabajo basado en **pseudo-etiquetado**:
1. **Adquisición**: Descargamos audios crudos en formato `.mp3` de la plataforma de ciencia ciudadana *Xeno-Canto*, filtrados para Colombia.
2. **Pseudo-Anotación**: Procesamos estos audios utilizando el modelo *BirdNET* de Cornell, un detector acústico robusto, para extraer segmentos con alta confianza donde se detecta canto activo.
3. **Modelado**: El modelo final (una Red Neuronal Convolucional Recurrente o CRNN) se entrenará para imitar y generalizar estas anotaciones a nivel de fotograma (*frame-level*).

### Especies Objetivo
Nos enfocamos en 10 de las especies de aves más comunes y acústicamente activas en Medellín (Valle de Aburrá):
* *Turdus ignobilis* (Mirla)
* *Pitangus sulphuratus* (Bichofué)
* *Thraupis episcopus* (Azulejo común)
* *Zonotrichia capensis* (Copetón)
* *Tyrannus melancholicus* (Sirirí)
* *Pygochelidon cyanoleuca* (Golondrina barranquera)
* *Troglodytes aedon* (Cucarachero)
* *Crotophaga ani* (Garrapatero)
* *Thraupis palmarum* (Azulejo palmero)
* *Campylorhynchus griseus* (Cucarachero chupahuevo)

## 3. Selección y Definición de las 10 Especies Objetivo

Para garantizar la viabilidad del detector acústico, seleccionamos 10 especies de aves de gran abundancia y representatividad acústica en la zona urbana del Valle de Aburrá (Medellín, Colombia). Esta selección nos permite acotar el vocabulario del modelo a clases con alta presencia de vocalizaciones en la base de datos abierta de *Xeno-Canto*, facilitando la recopilación de datos con buena relación señal-ruido.

In [2]:
especies_info = {
    "Turdus ignobilis": {
        "nombre_comun": "Mirla / Mayo",
        "nombre_ingles": "Black-billed Thrush",
        "justificacion": "Muy abundante en parques urbanos. Canto territorial continuo, melodioso y potente, con picos de actividad al amanecer."
    },
    "Pitangus sulphuratus": {
        "nombre_comun": "Bichofué / Bienteveo",
        "nombre_ingles": "Great Kiskadee",
        "justificacion": "Muy adaptado a entornos urbanos. Su vocalización onomatopéyica ('bicho-fué') es estridente, repetitiva y de fácil detección."
    },
    "Thraupis episcopus": {
        "nombre_comun": "Azulejo Común",
        "nombre_ingles": "Blue-gray Tanager",
        "justificacion": "Omnipresente en el Valle de Aburrá. Sus vocalizaciones consisten en silbidos agudos, rápidos y ligeramente chirriantes."
    },
    "Zonotrichia capensis": {
        "nombre_comun": "Copetón / Gorrión",
        "nombre_ingles": "Rufous-collared Sparrow",
        "justificacion": "El gorrión urbano por excelencia de los Andes colombianos. Canto territorial corto consistente en silbidos seguidos de un trino rápido."
    },
    "Tyrannus melancholicus": {
        "nombre_comun": "Sirirí / Suirirí",
        "nombre_ingles": "Tropical Kingbird",
        "justificacion": "Posado típicamente en cables de energía y postes. Emite series rápidas de trinos agudos y metálicos de carácter territorial."
    },
    "Pygochelidon cyanoleuca": {
        "nombre_comun": "Golondrina Barranquera",
        "nombre_ingles": "Blue-and-white Swallow",
        "justificacion": "Frecuenta bordes de construcciones y quebradas urbanas. Sus vocalizaciones consisten en zumbidos rápidos emitidos frecuentemente al vuelo."
    },
    "Troglodytes aedon": {
        "nombre_comun": "Cucarachero Común",
        "nombre_ingles": "House Wren",
        "justificacion": "Pequeño y ruidoso, habita jardines y rendijas de estructuras. Posee un canto complejo de cascadas de gorjeos rápidos y trinos intensos."
    },
    "Crotophaga ani": {
        "nombre_comun": "Garrapatero / Garrapatuelo",
        "nombre_ingles": "Smooth-billed Ani",
        "justificacion": "Habita zonas verdes abiertas y parches de maleza urbana. Emiten silbidos quejumbrosos, agudos y ascendentes de manera gregaria."
    },
    "Thraupis palmarum": {
        "nombre_comun": "Azulejo Palmero",
        "nombre_ingles": "Palm Tanager",
        "justificacion": "Estrechamente asociado a palmeras urbanas. Sus cantos son chirriantes y rápidos, planteando un reto de clasificación frente a T. episcopus."
    },
    "Campylorhynchus griseus": {
        "nombre_comun": "Cucarachero Chupahuevo / Matraca",
        "nombre_ingles": "Bicolored Wren",
        "justificacion": "Frecuente en árboles ornamentales densos. Emite vocalizaciones ásperas y sumamente ruidosas coordinadas en grupos familiares."
    }
}

# Generar lista plana para búsquedas rápidas en la API de Xeno-Canto
especies_medellin = list(especies_info.keys())

# Mostrar la tabla descriptiva como un DataFrame de Pandas para la documentación del notebook
df_especies = pd.DataFrame.from_dict(especies_info, orient="index")
df_especies.index.name = "Nombre Científico"
df_especies.reset_index(inplace=True)
display(df_especies)

,Nombre Científico,nombre_comun,nombre_ingles,justificacion
0,Turdus ignobilis,Mirla / Mayo,Black-billed Thrush,Muy abundante en parques urbanos. Canto territ...
1,Pitangus sulphuratus,Bichofué / Bienteveo,Great Kiskadee,Muy adaptado a entornos urbanos. Su vocalizaci...
2,Thraupis episcopus,Azulejo Común,Blue-gray Tanager,Omnipresente en el Valle de Aburrá. Sus vocali...
3,Zonotrichia capensis,Copetón / Gorrión,Rufous-collared Sparrow,El gorrión urbano por excelencia de los Andes ...
4,Tyrannus melancholicus,Sirirí / Suirirí,Tropical Kingbird,Posado típicamente en cables de energía y post...
5,Pygochelidon cyanoleuca,Golondrina Barranquera,Blue-and-white Swallow,Frecuenta bordes de construcciones y quebradas...
6,Troglodytes aedon,Cucarachero Común,House Wren,"Pequeño y ruidoso, habita jardines y rendijas ..."
7,Crotophaga ani,Garrapatero / Garrapatuelo,Smooth-billed Ani,Habita zonas verdes abiertas y parches de male...
8,Thraupis palmarum,Azulejo Palmero,Palm Tanager,Estrechamente asociado a palmeras urbanas. Sus...
9,Campylorhynchus griseus,Cucarachero Chupahuevo / Matraca,Bicolored Wren,Frecuente en árboles ornamentales densos. Emit...


## 4. Implementación de la Descarga de Audios desde Xeno-Canto

Para obtener las señales de audio consumiremos la API pública de **Xeno-Canto (Versión 3)**. La consulta se estructura filtrando específicamente por especie (`sp:"Nombre Científico"`) y restringiendo la localización geográfica a Colombia (`cnt:colombia`) con el fin de priorizar variaciones y modulaciones acústicas locales.

Para respetar los límites de la API de Xeno-Canto y evitar bloqueos por tasa de peticiones, la función implementa un retardo forzado de un segundo (`time.sleep(1)`) entre descargas exitosas, además de una verificación local para evitar re-descargar archivos que ya existan en el sistema.

In [3]:
# API Key oficial provista por Xeno-Canto
API_KEY = "f803b7018e2a9638e0c9c50998f681bcb1b1042f"

def descargar_de_xeno_canto(especie_nombre, max_archivos=120, base_dir="../dataset_aves"):
    """
    Busca grabaciones de una especie en la API de Xeno-Canto v3 filtradas por país (Colombia)
    y las descarga organizadamente en carpetas locales en formato MP3.
    """
    nombre_carpeta = especie_nombre.replace(" ", "_")
    carpeta_especie = os.path.join(base_dir, nombre_carpeta)
    os.makedirs(carpeta_especie, exist_ok=True)

    print(f"\nBuscando en API de Xeno-Canto: {especie_nombre} ...")

    # Usamos espacio para separar criterios y pasamos parámetros de forma limpia a requests
    params = {
        "query": f'sp:"{especie_nombre}" cnt:colombia',
        "key": API_KEY
    }
    url = "https://xeno-canto.org/api/3/recordings"

    try:
        # requests.get codificará automáticamente los espacios como '+' de forma válida
        respuesta = requests.get(url, params=params, timeout=15)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"  [Error] Fallo en la petición de búsqueda a Xeno-Canto: {e}")
        return

    datos = respuesta.json()
    grabaciones = datos.get('recordings', [])
    cant_disponible = len(grabaciones)
    descargas = min(max_archivos, cant_disponible)

    if descargas == 0:
        print("  - No se hallaron grabaciones que cumplan el criterio geográfico para esta especie.")
        return

    print(f"  - Grabaciones disponibles: {cant_disponible}. Cantidad a descargar: {descargas}...")

    guardados_exito = 0
    for grabacion in grabaciones[:descargas]:
        audio_url = grabacion.get('file')
        if not audio_url:
            continue

        ruta_archivo = os.path.join(carpeta_especie, f"{grabacion['id']}.mp3")

        if not os.path.exists(ruta_archivo):
            try:
                res_audio = requests.get(audio_url, timeout=30)
                if res_audio.status_code == 200:
                    with open(ruta_archivo, 'wb') as f:
                        f.write(res_audio.content)
                    guardados_exito += 1
                
                # Pausa prudencial entre descargas consecutivas
                time.sleep(1)
            except Exception as e:
                print(f"  [Error] No se pudo descargar la grabación ID {grabacion['id']}: {e}")
        else:
            guardados_exito += 1

    print(f"  - Completado: {guardados_exito}/{descargas} archivos en '{carpeta_especie}'.")

## 5. Extracción y Compilación de Metadatos Científicos

Para garantizar el rigor metodológico del dataset, recopilaremos y estructuraremos un registro de metadatos completo por cada audio gestionado. Esto nos permitirá auditar sesgos geográficos, de calidad o de autoría antes del modelado estadístico.

Las variables que extraeremos de cada registro JSON son:
*   `id_audio`: Identificador numérico único de la grabación en Xeno-Canto.
*   `especie_esperada`: Nombre de la carpeta objetivo de la descarga.
*   `genero` / `especie_nombre`: Taxonomía oficial registrada.
*   `autor` / `licencia`: Propiedad intelectual y tipo de licencia de uso.
*   `pais` / `localidad` / `latitud` / `longitud`: Contexto geográfico del registro.
*   `calidad`: Calificación de la grabación (escala A a E asignada por la comunidad).
*   `duracion_original`: Duración del archivo de audio original antes del procesamiento.
*   `formato`: Extensión del archivo original (típicamente MP3).
*   `tipo_vocalizacion`: Tipo de sonido registrado (canto, llamado, etc.).
*   `url_descarga`: Enlace directo de descarga.

In [4]:
# Inicializar lista para almacenar los diccionarios de metadatos
metadatos_audios = []

# Ajustar volumen de descarga (puedes cambiarlo a 120 para el dataset completo de entrenamiento)
MAX_ARCHIVOS_POR_ESPECIE = 120 

print(f"Iniciando descarga física y compilación de metadatos de audios...")
print(f"Configuración actual: Máximo {MAX_ARCHIVOS_POR_ESPECIE} audios por especie.")

for especie in especies_medellin:
    # 1. Invocar la descarga física utilizando la nueva versión corregida con params
    descargar_de_xeno_canto(especie, max_archivos=MAX_ARCHIVOS_POR_ESPECIE, base_dir=BASE_DIR)
    
    # 2. Consultar la API usando parámetros limpios para recopilar los metadatos de las mismas descargas
    params = {
        "query": f'sp:"{especie}" cnt:colombia',
        "key": API_KEY
    }
    url = "https://xeno-canto.org/api/3/recordings"
    
    try:
        respuesta = requests.get(url, params=params, timeout=15)
        if respuesta.status_code == 200:
            grabaciones = respuesta.json().get('recordings', [])
            
            for grabacion in grabaciones[:MAX_ARCHIVOS_POR_ESPECIE]:
                metadatos_audios.append({
                    "id_audio": grabacion.get("id"),
                    "especie_esperada": especie,
                    "genero": grabacion.get("gen"),
                    "especie_nombre": grabacion.get("sp"),
                    "nombre_ingles": grabacion.get("en"),
                    "autor": grabacion.get("rec"),
                    "pais": grabacion.get("cnt"),
                    "localidad": grabacion.get("loc"),
                    "latitud": grabacion.get("lat"),
                    "longitud": grabacion.get("lng"),
                    "calidad": grabacion.get("q"),
                    "duracion_original": grabacion.get("length"),
                    "licencia": grabacion.get("lic"),
                    "formato": grabacion.get("fmt"),
                    "url_descarga": grabacion.get("file"),
                    "tipo_vocalizacion": grabacion.get("type"),
                    "fecha_registro": grabacion.get("date")
                })
    except Exception as e:
        print(f"  [Advertencia] No se pudieron compilar metadatos para {especie}: {e}")

# Construir el DataFrame de metadatos a partir de la lista acumulada
df_metadata_completa = pd.DataFrame(metadatos_audios)

print(f"\nProceso finalizado.")
print(f"Total de registros de metadatos almacenados en memoria: {len(df_metadata_completa)}")

# Mostrar una muestra del DataFrame resultante si no está vacío
if not df_metadata_completa.empty:
    display(df_metadata_completa.head(5))

Iniciando descarga física y compilación de metadatos de audios...
Configuración actual: Máximo 120 audios por especie.

Buscando en API de Xeno-Canto: Turdus ignobilis ...
  - Grabaciones disponibles: 62. Cantidad a descargar: 62...
  - Completado: 62/62 archivos en '../dataset_aves/Turdus_ignobilis'.

Buscando en API de Xeno-Canto: Pitangus sulphuratus ...
  - Grabaciones disponibles: 100. Cantidad a descargar: 100...
  - Completado: 100/100 archivos en '../dataset_aves/Pitangus_sulphuratus'.

Buscando en API de Xeno-Canto: Thraupis episcopus ...
  - Grabaciones disponibles: 71. Cantidad a descargar: 71...
  - Completado: 71/71 archivos en '../dataset_aves/Thraupis_episcopus'.

Buscando en API de Xeno-Canto: Zonotrichia capensis ...
  - Grabaciones disponibles: 100. Cantidad a descargar: 100...
  - Completado: 100/100 archivos en '../dataset_aves/Zonotrichia_capensis'.

Buscando en API de Xeno-Canto: Tyrannus melancholicus ...
  - Grabaciones disponibles: 86. Cantidad a descargar: 86.

,id_audio,especie_esperada,genero,especie_nombre,nombre_ingles,autor,pais,localidad,latitud,longitud,calidad,duracion_original,licencia,formato,url_descarga,tipo_vocalizacion,fecha_registro
0,742378,Turdus ignobilis,Turdus,ignobilis,Black-billed Thrush,Joost van Bruggen,Colombia,"Victoria, Caldas",5.3191,None,A,0:50,https://creativecommons.org/licenses/by-nc-nd/...,None,https://xeno-canto.org/742378/download,call,2022-07-03
1,742377,Turdus ignobilis,Turdus,ignobilis,Black-billed Thrush,Joost van Bruggen,Colombia,"Victoria, Caldas",5.3191,None,A,0:04,https://creativecommons.org/licenses/by-nc-nd/...,None,https://xeno-canto.org/742377/download,alarm call,2022-07-03
2,683965,Turdus ignobilis,Turdus,ignobilis,Black-billed Thrush,Mauricio Cuellar Ramirez (@Birding.travel),Colombia,"Pava Negra Nature Reserve - Florencia, Caquetá.",1.7135,None,A,0:22,https://creativecommons.org/licenses/by-nc-sa/...,None,https://xeno-canto.org/683965/download,"song, llamadas de agresión",2021-10-25
3,614016,Turdus ignobilis,Turdus,ignobilis,Black-billed Thrush,Mauricio Cuellar Ramirez (@Birding.travel),Colombia,"Finca San Jorge, Morelia/Caquetá",1.4898,None,A,0:44,https://creativecommons.org/licenses/by-nc-sa/...,None,https://xeno-canto.org/614016/download,song,2021-01-07
4,576274,Turdus ignobilis,Turdus,ignobilis,Black-billed Thrush,Jerome Fischer,Colombia,"Ukuku Rural Lodge, Juntas, Ibague, Tolima, Co...",4.5585,None,A,1:02,https://creativecommons.org/licenses/by-nc-sa/...,None,https://xeno-canto.org/576274/download,song,2020-07-15


## 6. Validación de Integridad Física y Decodificación de Audios

Un reto común al trabajar con audios distribuidos de forma abierta (como los de *Xeno-Canto*) es la presencia de archivos MP3 corruptos debido a subidas interrumpidas, formatos de tasa de bits variable no estandarizados o problemas en los códecs locales del servidor de origen. Si estos audios corruptos se ignoran, durante el modelado la librería de procesamiento fallará silenciosamente, forzando al cargador de datos a retornar matrices vacías (llesnas de ceros), lo que confunde severamente el entrenamiento de la red neuronal [5].{}+[Cexrgfolkdmñasz]

Para mitigar esto de raíz, implementaremos una **sanitización preventiva estricta**:
1. Intentaremos cargar y decodificar por completo cada archivo `.mp3` descargado utilizando `torchaudio.load()`.
2. Mediremos las propiedades acústicas de la señal real decodificada: número de canales, tasa de muestreo (*sample rate*) real y duración real medida en segundos.
3. Si la decodificación falla y lanza cualquier excepción (como errores de códecs o de lectura de tramas), **eliminaremos el archivo físicamente de la máquina** y descartaremos permanentemente su registro del DataFrame de metadatos, asegurando un dataset completamente funcional [5].

In [5]:
# Lista para registrar los índices de metadatos que presenten problemas y deban eliminarse
indices_descartar = []

# Listas para almacenar las dimensiones físicas reales medidas directamente del códec
duraciones_reales = []
sample_rates_reales = []
canales_reales = []

print("Iniciando validación física de decodificación en audios descargados...")

for idx, row in tqdm(df_metadata_completa.iterrows(), total=len(df_metadata_completa)):
    nombre_carpeta = row["especie_esperada"].replace(" ", "_")
    ruta_archivo = os.path.join(BASE_DIR, nombre_carpeta, f"{row['id_audio']}.mp3")
    
    # 1. Comprobación de existencia del archivo en disco
    if not os.path.exists(ruta_archivo):
        print(f"\n  [Advertencia] ID {row['id_audio']} ({row['especie_esperada']}) no está presente en la ruta física.")
        indices_descartar.append(idx)
        
        # CORRECCIÓN DE BUG: Añadir None para mantener el alineamiento antes del drop
        duraciones_reales.append(None)
        sample_rates_reales.append(None)
        canales_reales.append(None)
        continue
        
    # 2. Intento de decodificación física del stream de audio
    try:
        # Intentamos decodificar las tramas completas del archivo con torchaudio
        waveform, sr = torchaudio.load(ruta_archivo)
        
        # Extraer metadatos acústicos reales medidos directamente
        num_canales = waveform.shape[0]
        num_frames = waveform.shape[1]
        duracion_segundos = num_frames / sr
        
        # Almacenar las propiedades medidas
        duraciones_reales.append(duracion_segundos)
        sample_rates_reales.append(sr)
        canales_reales.append(num_canales)
        
    except Exception as e:
        # Reportar fallo y proceder a eliminar el archivo corrupto del almacenamiento local
        print(f"\n  [Archivo Corrupto] Falló decodificación de ID {row['id_audio']} ({row['especie_esperada']}).")
        print(f"  -> Causa: {e}")
        print(f"  -> Eliminando archivo corrupto de forma física del disco...")
        
        # Eliminar archivo físico para no contaminar pipelines posteriores
        try:
            os.remove(ruta_archivo)
        except Exception as err_del:
            print(f"    [Error] No se pudo remover el archivo del disco: {err_del}")
            
        indices_descartar.append(idx)
        
        # Llenar con nulos para mantener alineadas las listas antes del drop
        duraciones_reales.append(None)
        sample_rates_reales.append(None)
        canales_reales.append(None)

# 3. Incorporar los metadatos acústicos reales validados al DataFrame principal
df_metadata_completa["duracion_real"] = duraciones_reales
df_metadata_completa["sample_rate_real"] = sample_rates_reales
df_metadata_completa["canales_real"] = canales_reales

# 4. Descartar definitivamente los registros de metadatos de audios corruptos o ausentes
if indices_descartar:
    df_metadata_completa.drop(indices_descartar, inplace=True)
    df_metadata_completa.reset_index(drop=True, inplace=True)
    print(f"\nSaneamiento completado: Se eliminaron {len(indices_descartar)} registros corruptos o ausentes.")
else:
    print("\nÉxito: Todos los audios se decodificaron exitosamente sin errores.")

print(f"Registros válidos y verificados restantes en el DataFrame: {len(df_metadata_completa)}")

Iniciando validación física de decodificación en audios descargados...


  3%|▎         | 25/755 [00:02<00:43, 16.96it/s]


  [Archivo Corrupto] Falló decodificación de ID 433294 (Turdus ignobilis).
  -> Causa: Failed to decode audio samples: decodeAVFrame, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:1495, Could not push packet to decoder: Invalid data found when processing input
  -> Eliminando archivo corrupto de forma física del disco...


  4%|▍         | 30/755 [00:03<00:58, 12.31it/s]


  [Archivo Corrupto] Falló decodificación de ID 425787 (Turdus ignobilis).
  -> Causa: Failed to decode audio samples: decodeAVFrame, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:1495, Could not push packet to decoder: Invalid data found when processing input
  -> Eliminando archivo corrupto de forma física del disco...


...


  [Archivo Corrupto] Falló decodificación de ID 393268 (Pitangus sulphuratus).
  -> Causa: Failed to decode audio samples: decodeAVFrame, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:1495, Could not push packet to decoder: Invalid data found when processing input
  -> Eliminando archivo corrupto de forma física del disco...


 63%|██████▎   | 476/755 [00:23<00:09, 28.58it/s]


  [Archivo Corrupto] Falló decodificación de ID 773325 (Troglodytes aedon).
  -> Causa: Failed to decode audio samples: decodeAVFrame, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:1495, Could not push packet to decoder: Invalid data found when processing input
  -> Eliminando archivo corrupto de forma física del disco...


100%|██████████| 755/755 [00:34<00:00, 21.83it/s]


Saneamiento completado: Se eliminaron 4 registros corruptos o ausentes.
Registros válidos y verificados restantes en el DataFrame: 751


## 7. Generación de Pseudo-Etiquetas Temporales con BirdNET

Una vez que hemos garantizado que los 50 archivos de audio del dataset de prueba son físicamente legibles, procedemos a realizar el **pseudo-etiquetado temporal** [5]. Utilizaremos la librería `birdnetlib`, que actúa como interfaz local para el modelo de aprendizaje profundo *BirdNET* desarrollado por el *Laboratorio de Ornitología de Cornell*.

El analizador procesará cada audio completo de forma secuencial y detectará la presencia de cantos de aves en ventanas de tiempo específicas. Configuraremos un umbral de confianza mínimo intencionalmente bajo (`min_conf=0.10`) para registrar vocalizaciones sutiles o distantes que un umbral estricto omitiría. Esto es vital para SED, ya que nos permitirá contar con etiquetas temporales de cantos tenues y posteriormente aplicar filtros más estrictos si así lo deseamos.

Este proceso generará nuestra matriz base de etiquetas temporales (con marcas de `inicio_segundo` y `fin_segundo`).

In [6]:
# 1. Inicializar el analizador de BirdNET (descarga automáticamente el modelo de Cornell si es la primera ejecución)
print("Inicializando modelo de BirdNET (Analyzer)...")
analyzer = Analyzer()
print("Modelo de BirdNET cargado exitosamente.")

# Lista para acumular los eventos acústicos detectados por BirdNET
detecciones_acumuladas = []

print("\nIniciando análisis acústico sobre los audios verificados...")

for idx, row in tqdm(df_metadata_completa.iterrows(), total=len(df_metadata_completa)):
    nombre_carpeta = row["especie_esperada"].replace(" ", "_")
    nombre_archivo = f"{row['id_audio']}.mp3"
    ruta_archivo = os.path.join(BASE_DIR, nombre_carpeta, nombre_archivo)
    
    # Asegurar que el archivo físico exista en disco antes de mandarlo a analizar
    if not os.path.exists(ruta_archivo):
        continue
        
    try:
        # Instanciar el objeto de grabación de BirdNET
        recording = Recording(analyzer, ruta_archivo, min_conf=0.10)
        recording.analyze()
        
        # Extraer cada detección encontrada dentro de la grabación
        for deteccion in recording.detections:
            detecciones_acumuladas.append({
                "archivo_original": nombre_archivo,
                "especie_esperada": row["especie_esperada"],  # Carpeta de origen
                "ave_detectada_comun": deteccion.get("common_name"),
                "ave_detectada_cientifico": deteccion.get("scientific_name"),
                "inicio_segundo": deteccion.get("start_time"),
                "fin_segundo": deteccion.get("end_time"),
                "confianza_birdnet": deteccion.get("confidence")
            })
            
    except Exception as e:
        print(f"\n  [Error] Falló el análisis de BirdNET para el audio ID {row['id_audio']}: {e}")

# Convertir la lista de diccionarios en el DataFrame de etiquetas fuertes
df_etiquetas_fuertes = pd.DataFrame(detecciones_acumuladas)

print(f"\nAnálisis acústico finalizado.")
print(f"Total de segmentos de cantos (pseudo-etiquetas) detectados: {len(df_etiquetas_fuertes)}")

# Mostrar una muestra de las detecciones encontradas
if not df_etiquetas_fuertes.empty:
    display(df_etiquetas_fuertes.head(5))

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Inicializando modelo de BirdNET (Analyzer)...
Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.
Modelo de BirdNET cargado exitosamente.

Iniciando análisis acústico sobre los audios verificados...


  0%|          | 0/751 [00:00<?, ?it/s]

read_audio_data
read_audio_data: complete, read  17 chunks.
analyze_recording 742378.mp3


  0%|          | 2/751 [00:03<16:18,  1.31s/it]

...

100%|█████████▉| 750/751 [13:15<00:03,  3.75s/it]

read_audio_data
read_audio_data: complete, read  71 chunks.
analyze_recording 367892.mp3


100%|██████████| 751/751 [13:21<00:00,  1.07s/it]


Análisis acústico finalizado.
Total de segmentos de cantos (pseudo-etiquetas) detectados: 10951


,archivo_original,especie_esperada,ave_detectada_comun,ave_detectada_cientifico,inicio_segundo,fin_segundo,confianza_birdnet
0,742378.mp3,Turdus ignobilis,Spectacled Parrotlet,Forpus conspicillatus,3.0,6.0,0.106267
1,742378.mp3,Turdus ignobilis,Spectacled Parrotlet,Forpus conspicillatus,6.0,9.0,0.232879
2,742378.mp3,Turdus ignobilis,Black-billed Thrush,Turdus ignobilis,6.0,9.0,0.208953
3,742378.mp3,Turdus ignobilis,Black-billed Thrush,Turdus ignobilis,9.0,12.0,0.391466
4,742378.mp3,Turdus ignobilis,Spectacled Parrotlet,Forpus conspicillatus,12.0,15.0,0.678364


## 8. El Concepto de Pseudo-Ground-Truth e Implicaciones de la Polifonía Acústica

### 8.1. Paisajes Sonoros Reales y Polifonía Acústica
En las ciencias bioacústicas, los registros de campo son inherentemente ruidosos e interactivos. A pesar de que los audios de *Xeno-Canto* son etiquetados por un recolector humano basándose en una especie "focal" (el ave a la que se dirigía el micrófono), el entorno natural es **polifónico**. Esto significa que es común que múltiples individuos de diferentes especies vocalicen de manera simultánea o superpuesta en el tiempo.

Al configurar el analizador de BirdNET con un umbral de sensibilidad bajo (`min_conf=0.10`), permitimos que el modelo débil actúe como un anotador multi-etiqueta activo, detectando cantos de fondo de otras especies colaterales presentes en el mismo archivo.

### 8.2. Implicaciones del Pseudo-Ground-Truth
El conjunto de etiquetas que utilizaremos para entrenar nuestra Red Neuronal Convolucional Recurrente (CRNN) no constituye una verdad de campo biológica absoluta (*ground-truth* generada por ornitólogos humanos), sino una aproximación sistemática generada por un modelo automatizado, denominada **pseudo-ground-truth** [5]. 

Esto introduce ciertas consideraciones estadísticas para nuestra evaluación:
1.  **Imitación de modelo (Distillation)**: El modelo final aprenderá e imitará los criterios y sesgos sistemáticos del clasificador de Cornell. El mAP de validación/test medirá la concordancia de nuestro modelo con las anotaciones de BirdNET, no necesariamente con la realidad biológica pura [5].
2.  **Modelado Multi-Etiqueta**: Dado que múltiples especies pueden vocalizar al mismo tiempo en un mismo frame, el problema no puede modelarse usando funciones de activación excluyentes como *Softmax* con pérdidas de *Categorical Cross Entropy*. En su lugar, el problema se tratará como $C$ problemas de clasificación binaria independientes mediante el uso de la función de activación *Sigmoide* por frame y especie, optimizada con la pérdida de **Entropía Cruzada Binaria (BCE)**.

## 9. Resolución del Bug de Asignación y Filtrado de Especies Objetivo

Como demostraron los resultados crudos del paso 1.7, un audio grabado y catalogado originalmente bajo una especie focal (como `Turdus ignobilis`) frecuentemente captura cantos ambientales de especies ajenas a nuestro listado (como `Forpus conspicillatus`) [5]. 

Si utilizáramos la lógica del notebook original de etiquetar basándonos en la carpeta de origen (`especie_esperada`), estaríamos entrenando al modelo con falsos positivos severos. Para solucionar esta inconsistencia de raíz, ejecutaremos dos acciones de control:
1.  **Filtrado Estricto por Vocabulario**: Descartaremos cualquier detección de BirdNET cuyo nombre científico (`ave_detectada_cientifico`) no pertenezca estrictamente a nuestra lista de 10 especies objetivo de Medellín.
2.  **Columna de Destino Unificada (`especie_target`)**: Crearemos explícitamente la columna `especie_target` igual a `ave_detectada_cientifico`. El cargador de datos del Notebook 03 leerá esta columna de forma exclusiva, rompiendo cualquier dependencia de la carpeta física donde se encuentra almacenado el archivo original, garantizando la fidelidad de las etiquetas temporales [5].

In [7]:
# 1. Medir el volumen inicial de detecciones crudas
total_inicial = len(df_etiquetas_fuertes)

# 2. Filtrar el DataFrame de etiquetas para mantener únicamente nuestras 10 especies
df_etiquetas_filtradas = df_etiquetas_fuertes[
    df_etiquetas_fuertes["ave_detectada_cientifico"].isin(especies_medellin)
].copy()

# 3. Crear explícitamente la columna de mapeo unificado para el modelo final
df_etiquetas_filtradas["especie_target"] = df_etiquetas_filtradas["ave_detectada_cientifico"]

# 4. Calcular el balance resultante
total_final = len(df_etiquetas_filtradas)
descartados = total_inicial - total_final

print("Saneamiento de etiquetas y resolución de bug completado:")
print(f"  - Detecciones crudas de BirdNET: {total_inicial}")
print(f"  - Segmentos descartados (especies de fondo no objetivo): {descartados}")
print(f"  - Segmentos válidos conservados (nuestras 10 especies): {total_final}")

# Mostrar cuántos segmentos reales tenemos para cada una de las especies objetivo en este lote de prueba
print("\nDistribución de segmentos de cantos válidos por especie objetivo:")
print(df_etiquetas_filtradas["especie_target"].value_counts())

# Reemplazar el DataFrame original con la versión ya saneada y filtrada
df_etiquetas_fuertes = df_etiquetas_filtradas.reset_index(drop=True)

# Mostrar los primeros registros de la tabla limpia de etiquetas fuertes
if not df_etiquetas_fuertes.empty:
    display(df_etiquetas_fuertes.head(5))

Saneamiento de etiquetas y resolución de bug completado:
  - Detecciones crudas de BirdNET: 10951
  - Segmentos descartados (especies de fondo no objetivo): 5310
  - Segmentos válidos conservados (nuestras 10 especies): 5641

Distribución de segmentos de cantos válidos por especie objetivo:
especie_target
Zonotrichia capensis       771
Turdus ignobilis           742
Pitangus sulphuratus       707
Thraupis episcopus         682
Crotophaga ani             593
Campylorhynchus griseus    573
Tyrannus melancholicus     492
Thraupis palmarum          475
Troglodytes aedon          371
Pygochelidon cyanoleuca    235
Name: count, dtype: int64


,archivo_original,especie_esperada,ave_detectada_comun,ave_detectada_cientifico,inicio_segundo,fin_segundo,confianza_birdnet,especie_target
0,742378.mp3,Turdus ignobilis,Black-billed Thrush,Turdus ignobilis,6.0,9.0,0.208953,Turdus ignobilis
1,742378.mp3,Turdus ignobilis,Black-billed Thrush,Turdus ignobilis,9.0,12.0,0.391466,Turdus ignobilis
2,742378.mp3,Turdus ignobilis,Black-billed Thrush,Turdus ignobilis,15.0,18.0,0.351664,Turdus ignobilis
3,742378.mp3,Turdus ignobilis,Black-billed Thrush,Turdus ignobilis,18.0,21.0,0.386466,Turdus ignobilis
4,742378.mp3,Turdus ignobilis,Black-billed Thrush,Turdus ignobilis,24.0,27.0,0.247040,Turdus ignobilis


## 10. Persistencia de Metadatos de Audios (`df_metadata_audios.csv`)

Una vez saneado el dataset físico de audios y extraídos los metadatos acústicos reales (como duración medida y tasa de muestreo real de códecs), procedemos a persistir esta información en el almacenamiento local [5].

Este archivo se guardará bajo la ruta `dataset_aves/df_metadata_audios.csv` y representará el inventario primario y registro de auditoría de nuestro corpus acústico [5].

In [8]:
# Guardar el DataFrame de metadatos de audios depurados en disco sin guardar el índice de pandas
df_metadata_completa.to_csv(METADATA_PATH, index=False)

print("Registro de metadatos de audios guardado con éxito:")
print(f"  - Ruta de destino: {os.path.abspath(METADATA_PATH)}")
print(f"  - Cantidad de audios válidos indexados: {len(df_metadata_completa)}")
print(f"  - Tamaño aproximado del archivo en disco: {os.path.getsize(METADATA_PATH) / 1024:.2f} KB")

Registro de metadatos de audios guardado con éxito:
  - Ruta de destino: /home/els4nchez/Pictures/Bird-Recognition-using-BioSED-model/dataset_aves/df_metadata_audios.csv
  - Cantidad de audios válidos indexados: 751
  - Tamaño aproximado del archivo en disco: 211.55 KB


## 11. Persistencia de Etiquetas Fuertes Saneadas (`df_etiquetas_fuertes.csv`)

Una vez que hemos resuelto el bug de asignación cruzada de especies por directorio y hemos filtrado los cantos de fondo para retener únicamente nuestras 10 especies objetivo de Medellín, procedemos a persistir la matriz de etiquetas fuertes en el disco [5].

Este archivo se guardará bajo la ruta `dataset_aves/df_etiquetas_fuertes.csv` [5]. Es el archivo más crítico para el modelado, ya que representa la "fuente de verdad débil" (*pseudo-ground-truth*) que el generador de lotes (*Data Loader*) consultará en el Notebook 03 para mapear de forma exacta qué frames de tiempo tienen cantos activos por especie [5].

In [9]:
# Guardar el DataFrame de etiquetas fuertes filtradas en disco sin guardar el índice de pandas
df_etiquetas_fuertes.to_csv(LABELS_PATH, index=False)

print("Registro de etiquetas fuertes guardado con éxito:")
print(f"  - Ruta de destino: {os.path.abspath(LABELS_PATH)}")
print(f"  - Cantidad de segmentos de canto indexados: {len(df_etiquetas_fuertes)}")
print(f"  - Tamaño aproximado del archivo en disco: {os.path.getsize(LABELS_PATH) / 1024:.2f} KB")

Registro de etiquetas fuertes guardado con éxito:
  - Ruta de destino: /home/els4nchez/Pictures/Bird-Recognition-using-BioSED-model/dataset_aves/df_etiquetas_fuertes.csv
  - Cantidad de segmentos de canto indexados: 5641
  - Tamaño aproximado del archivo en disco: 644.77 KB


## 12. Consolidación de Estadísticas Descriptivas del Dataset

Para auditar el balance y la distribución de datos antes de transferir el corpus al Notebook 02 (Análisis Exploratorio de Datos), consolidaremos un reporte descriptivo unificado. 

Este reporte cruzará la información del inventario de audios y del dataset de etiquetas fuertes, mostrando para cada una de las 10 especies objetivo:
1.  La cantidad de archivos de audio físicamente válidos en disco [5].
2.  La cantidad total de eventos de canto (*sound events*) temporalmente activos.
3.  La tasa promedio de eventos por archivo de audio (densidad acústica).

Adicionalmente, calcularemos una métrica global de control de calidad que registra cuántos audios corruptos debieron ser eliminados y depurados del sistema [5].

In [10]:
# 1. Contar la distribución física de audios válidos por especie
audios_validos_serie = df_metadata_completa["especie_esperada"].value_counts()

# 2. Contar la cantidad de segmentos de canto válidos por especie objetivo
eventos_validos_serie = df_etiquetas_fuertes["especie_target"].value_counts()

# 3. Consolidar ambas series en un DataFrame descriptivo estructurado
df_resumen = pd.DataFrame({
    "Audios Válidos": audios_validos_serie,
    "Eventos de Canto": eventos_validos_serie
})

# Saneamiento de nulos (por si alguna especie no registró cantos en este lote de prueba rápido)
df_resumen.fillna(0, inplace=True)
df_resumen = df_resumen.astype(int)

# Calcular métrica de densidad acústica derivada
df_resumen["Tasa Eventos/Audio"] = (df_resumen["Eventos de Canto"] / df_resumen["Audios Válidos"]).round(2)

df_resumen.index.name = "Especie"
df_resumen.reset_index(inplace=True)

# 4. Calcular consolidados globales de auditoría de descargas y descartes
total_intentados = len(especies_medellin) * MAX_ARCHIVOS_POR_ESPECIE
total_validos = len(df_metadata_completa)
total_descartados = total_intentados - total_validos

print("==============================================")
print("        REPORTE DE AUDITORÍA DEL DATASET      ")
print("==============================================")
print(f"Total de archivos de audio intentados: {total_intentados}")
print(f"Total de archivos de audio válidos:    {total_validos}")
print(f"Total de archivos de audio descartados: {total_descartados}")
print(f"Total de segmentos de cantos activos:   {df_resumen['Eventos de Canto'].sum()}")
print("==============================================\n")

# Mostrar la tabla descriptiva formateada
display(df_resumen)

        REPORTE DE AUDITORÍA DEL DATASET      
Total de archivos de audio intentados: 1200
Total de archivos de audio válidos:    751
Total de archivos de audio descartados: 449
Total de segmentos de cantos activos:   5641



,Especie,Audios Válidos,Eventos de Canto,Tasa Eventos/Audio
0,Campylorhynchus griseus,74,573,7.74
1,Crotophaga ani,83,593,7.14
2,Pitangus sulphuratus,99,707,7.14
3,Pygochelidon cyanoleuca,33,235,7.12
4,Thraupis episcopus,71,682,9.61
5,Thraupis palmarum,46,475,10.33
6,Troglodytes aedon,99,371,3.75
7,Turdus ignobilis,60,742,12.37
8,Tyrannus melancholicus,86,492,5.72
9,Zonotrichia capensis,100,771,7.71


## 13. Verificación Final del Dataset y Transición a Volumen Académico Completo

### 13.1. Verificación Programática de Estructuras
Antes de dar por concluida la ejecución, correremos una auditoría programática final. Esta celda inspeccionará el sistema de archivos local para certificar que contamos con los índices de metadatos, tablas de etiquetas fuertes y las carpetas de audio requeridas de forma unificada para los Notebooks 02 (EDA), 03 (Entrenamiento), 04 (Evaluación) y 05 (Dashboard Demo) [5].

### 13.2. Recordatorio: Transición al Volumen de Entrenamiento Real (120 Audios)
Hasta este momento, este notebook se ha ejecutado bajo un modo de desarrollo controlado con `MAX_ARCHIVOS_POR_ESPECIE = 5` para garantizar que la depuración de dependencias, la API de Xeno-Canto y las pseudo-etiquetas de BirdNET fluyan sinNameErrors ni bloqueos de red [5].

Para generar el conjunto de datos de entrenamiento real para la CRNN de tu compañero:
1.  Sube a la celda del **Paso 1.5** de este notebook.
2.  Modifica la variable: `MAX_ARCHIVOS_POR_ESPECIE = 120`.
3.  En la barra superior de Jupyter, selecciona **Kernel -> Restart & Run All** (Reiniciar y Ejecutar todo).
4.  El notebook se ejecutará de forma completamente automatizada de principio a fin, descargará los 1200 archivos de audio de Xeno-Canto, aplicará el filtro preventivo de decodificación, ejecutará la extracción de BirdNET de todos ellos y persistirá los archivos `.csv` finales listos para el modelado real [5].

In [11]:
# Inicializar contador de inconsistencias
errores_encontrados = 0

print("=== INICIANDO AUDITORÍA AUTOMATIZADA DE CONSISTENCIA ===")

# 1. Comprobar archivo de metadatos de audio
if os.path.exists(METADATA_PATH):
    df_test_meta = pd.read_csv(METADATA_PATH)
    if len(df_test_meta) > 0:
        print(f"  [OK] df_metadata_audios.csv presente. Registros indexados: {len(df_test_meta)}")
    else:
        print("  [ERROR] df_metadata_audios.csv se encuentra vacío en disco.")
        errores_encontrados += 1
else:
    print("  [ERROR] No se encuentra el archivo df_metadata_audios.csv en la ruta esperada.")
    errores_encontrados += 1

# 2. Comprobar archivo de etiquetas fuertes filtradas
if os.path.exists(LABELS_PATH):
    df_test_labels = pd.read_csv(LABELS_PATH)
    if len(df_test_labels) > 0:
        print(f"  [OK] df_etiquetas_fuertes.csv presente. Segmentos de canto indexados: {len(df_test_labels)}")
    else:
        print("  [ERROR] df_etiquetas_fuertes.csv se encuentra vacío en disco.")
        errores_encontrados += 1
else:
    print("  [ERROR] No se encuentra el archivo df_etiquetas_fuertes.csv en la ruta esperada.")
    errores_encontrados += 1

# 3. Comprobar carpetas físicas de especies e inventario real de audios MP3 en disco
carpetas_especie_encontradas = 0
audios_totales_contados = 0

for especie in especies_medellin:
    nombre_carpeta = especie.replace(" ", "_")
    ruta_carpeta = os.path.join(BASE_DIR, nombre_carpeta)
    
    if os.path.isdir(ruta_carpeta):
        carpetas_especie_encontradas += 1
        archivos_mp3 = [f for f in os.listdir(ruta_carpeta) if f.endswith(".mp3")]
        audios_totales_contados += len(archivos_mp3)
    else:
        print(f"  [ERROR] No existe la carpeta de almacenamiento de audio para: {especie}")
        errores_encontrados += 1

print(f"  [OK] Subdirectorios taxonómicos localizados: {carpetas_especie_encontradas}/10")
print(f"  [OK] Archivos físicos de audio .mp3 detectados: {audios_totales_contados}")

print("\n=== DICTAMEN FINAL DE LA AUDITORÍA ===")
if errores_encontrados == 0:
    print("ESTADO: DATASET CONSISTENTE")
    print("El notebook ha generado toda la estructura física e índices de control necesarios de forma correcta.")
else:
    print(f"ESTADO: INCONSISTENTE. Se identificaron {errores_encontrados} inconsistencias que deben corregirse.")

=== INICIANDO AUDITORÍA AUTOMATIZADA DE CONSISTENCIA ===
  [OK] df_metadata_audios.csv presente. Registros indexados: 751
  [OK] df_etiquetas_fuertes.csv presente. Segmentos de canto indexados: 5641
  [OK] Subdirectorios taxonómicos localizados: 10/10
  [OK] Archivos físicos de audio .mp3 detectados: 751

=== DICTAMEN FINAL DE LA AUDITORÍA ===
ESTADO: DATASET CONSISTENTE
El notebook ha generado toda la estructura física e índices de control necesarios de forma correcta.
